# Notebook 04 — Model Baselines

**Project:** Detecting Service Gaps and Bias Signals in Hospital Reviews  

---

## Goals
1. Build a TF-IDF + Logistic Regression baseline sentiment classifier
2. Compare it against the lexicon-only heuristic classifier
3. Evaluate with accuracy, F1, confusion matrix, and ROC curve
4. Analyse error cases to find where the model fails — these are the most interesting reviews

## 0 — Setup

In [ ]:
import sys
import os
from pathlib import Path

ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import re
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    roc_auc_score, roc_curve, confusion_matrix
)

matplotlib.rcParams['figure.dpi'] = 120

FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2

print('Setup complete.')

## 1 — Load Data

In [ ]:
flagged_path = ROOT / 'data' / 'processed' / 'reviews_flagged.csv'

if flagged_path.exists():
    df = pd.read_csv(flagged_path)
    print(f'Loaded flagged data: {df.shape}')
else:
    df = pd.read_csv(ROOT / 'data' / 'hospital.csv')
    df.columns = df.columns.str.strip()
    df = df.rename(columns={
        'Feedback':        'review_text',
        'Sentiment Label': 'sentiment_label',
        'Ratings':         'rating',
    })
    df = df.loc[:, ~df.columns.str.startswith('Unnamed')]
    df = df.drop_duplicates().dropna(subset=['review_text'])
    df['sentiment_label'] = pd.to_numeric(df['sentiment_label'], errors='coerce').fillna(0).astype(int)
    df['sentiment'] = df['sentiment_label'].map({1:'positive', 0:'negative'})

    CONCERN_LEXICON = {
        'mistreatment': ['rude','ignored','dismissive','disrespectful','insulted','negligent','unprofessional'],
        'access_barriers': ['wheelchair','ramp','accessible','lift','elevator','disabled','handicap','parking'],
        'delays_wait': ['waiting','wait','waited','hours','queue','delayed','slow','late','forever','long time'],
        'safety_hygiene': ['dirty','unclean','unhygienic','hygiene','infection','infested','smell','smelly','contaminated'],
    }
    for cat, kws in CONCERN_LEXICON.items():
        df[f'flag_{cat}'] = df['review_text'].apply(
            lambda x: int(any(kw in str(x).lower() for kw in kws)))
    flag_cols = [f'flag_{c}' for c in CONCERN_LEXICON]
    df['flag_any_concern'] = (df[flag_cols].sum(axis=1) > 0).astype(int)
    df = df.reset_index(drop=True)

X = df['review_text'].fillna('')
y = df['sentiment_label']

print(f'Label balance:\n{y.value_counts()}')

## 2 — Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
    X, y, df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Test label balance:\n{y_test.value_counts()}')

## 3 — TF-IDF + Logistic Regression Baseline

In [ ]:
# Vectorise
tfidf = TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), sublinear_tf=True)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

# Train classifier
clf = LogisticRegression(max_iter=500, random_state=RANDOM_STATE, class_weight='balanced')
clf.fit(X_train_tfidf, y_train)

# Predict
y_pred = clf.predict(X_test_tfidf)
y_prob = clf.predict_proba(X_test_tfidf)[:, 1]

acc  = accuracy_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred, average='weighted')
roc  = roc_auc_score(y_test, y_prob)

print(f'Accuracy : {acc:.4f}')
print(f'F1 (wtd) : {f1:.4f}')
print(f'ROC-AUC  : {roc:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['negative', 'positive']))

## 4 — Lexicon-Only Heuristic Baseline

In [ ]:
# Heuristic: if no concern flag → predict positive; if any flag → predict negative
flag_cols = [c for c in df.columns if c.startswith('flag_') and c != 'flag_any_concern']
df_test = df.loc[idx_test].copy()

y_lex = (df_test['flag_any_concern'] == 0).astype(int)  # 1=positive if no flag

acc_lex = accuracy_score(y_test, y_lex)
f1_lex  = f1_score(y_test, y_lex, average='weighted')

print(f'Lexicon Heuristic — Accuracy: {acc_lex:.4f}  |  F1 (wtd): {f1_lex:.4f}')
print(classification_report(y_test, y_lex, target_names=['negative', 'positive']))

## 5 — Comparison Table

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Lexicon Heuristic', 'TF-IDF + LR'],
    'Accuracy': [acc_lex, acc],
    'F1 (weighted)': [f1_lex, f1],
    'ROC-AUC': ['-', f'{roc:.4f}'],
    'Interpretable?': ['Yes', 'Partially'],
})
print(comparison.to_string(index=False))

## 6 — Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
tick_labels = ['negative', 'positive']
ax.set_xticks([0, 1]); ax.set_xticklabels(tick_labels)
ax.set_yticks([0, 1]); ax.set_yticklabels(tick_labels)
ax.set_xlabel('Predicted', fontsize=10)
ax.set_ylabel('Actual', fontsize=10)
ax.set_title('Confusion Matrix — TF-IDF + LR', fontsize=12, fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                fontsize=14, color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/confusion_matrix.png')

## 7 — ROC Curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr, tpr, color='#3498db', lw=2, label=f'TF-IDF + LR (AUC = {roc:.3f})')
ax.plot([0,1],[0,1], 'k--', lw=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — Sentiment Classification', fontsize=12, fontweight='bold')
ax.legend(loc='lower right')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'roc_curve.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/roc_curve.png')

## 8 — Top Predictive Features

In [ ]:
feature_names = tfidf.get_feature_names_out()
coefs = clf.coef_[0]
N = 15

top_pos_idx = coefs.argsort()[-N:][::-1]
top_neg_idx = coefs.argsort()[:N]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.barh(feature_names[top_pos_idx][::-1], coefs[top_pos_idx][::-1], color='#2ecc71', edgecolor='white')
ax1.set_title(f'Top {N} Features → Positive Sentiment', fontsize=11, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

ax2.barh(feature_names[top_neg_idx][::-1], coefs[top_neg_idx][::-1], color='#e74c3c', edgecolor='white')
ax2.set_title(f'Top {N} Features → Negative Sentiment', fontsize=11, fontweight='bold')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.suptitle('Logistic Regression — Most Influential Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_features.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/top_features.png')

## 9 — Error Analysis

Examine misclassified reviews — these often contain the most interesting edge cases: ambiguous language, sarcasm, code-switching.

In [ ]:
df_test = df.loc[idx_test].copy()
df_test['y_pred'] = y_pred
df_test['y_prob_positive'] = y_prob

errors = df_test[df_test['sentiment_label'] != df_test['y_pred']].copy()
print(f'Misclassified: {len(errors):,} / {len(df_test):,} ({len(errors)/len(df_test)*100:.1f}%)')

# False Negatives: actually positive, predicted negative
fn = errors[errors['sentiment_label'] == 1][['review_text','y_prob_positive']].head(3)
print('\nFalse Negatives (actually positive, predicted negative):')
for _, row in fn.iterrows():
    print(f'  [prob={row["y_prob_positive"]:.2f}] {str(row["review_text"])[:200]}')
    print()

# False Positives: actually negative, predicted positive
fp = errors[errors['sentiment_label'] == 0][['review_text','y_prob_positive']].head(3)
print('\nFalse Positives (actually negative, predicted positive):')
for _, row in fp.iterrows():
    print(f'  [prob={row["y_prob_positive"]:.2f}] {str(row["review_text"])[:200]}')
    print()

---
## Summary

| Model | Accuracy | F1 (wtd) | Interpretable? |
|-------|----------|----------|----------------|
| Lexicon Heuristic | see above | see above | ✅ Fully |
| TF-IDF + LR | see above | see above | ⚠️ Partially |

**Key insight**: The lexicon-only heuristic provides a strong interpretable baseline. The TF-IDF + LR model adds statistical power but at the cost of direct interpretability. For operational decisions, always pair the ML model's predictions with the lexicon flags to explain *why*.

**Error analysis** surfaced ambiguous reviews (sarcasm, hedged positives, code-switching) that no automated classifier handles well — this highlights the need for **human review** in high-stakes decisions.